In [ ]:
%%capture
!pip install transformers>=4.48.0 sentence-transformers faiss-cpu pdfplumber gradio peft bitsandbytes accelerate huggingface_hub -q

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download, login
import os

HF_TOKEN = "HF_API_KEY"
KULLANICI = "boranyzgc"

login(token=HF_TOKEN)

print("Modeller indiriliyor...")

mursit_path = snapshot_download(repo_id=f"{KULLANICI}/mursit-large-tur2-v2", token=HF_TOKEN)
print(f"✅ Mursit: {mursit_path}")

reranker_path = snapshot_download(repo_id=f"{KULLANICI}/bge-reranker-ft", token=HF_TOKEN)
print(f"✅ BGE Reranker: {reranker_path}")

adapter_path = snapshot_download(repo_id=f"{KULLANICI}/trendyol-ft-context", token=HF_TOKEN)
print(f"✅ Trendyol adapter: {adapter_path}")

mevzuat_path = hf_hub_download(
    repo_id=f"{KULLANICI}/turkish-legal-mevzuat",
    filename="mevzuat_chunked0v2_normalized.json",
    repo_type="dataset",
    token=HF_TOKEN
)
print(f"✅ Mevzuat: {mevzuat_path}")

Modeller indiriliyor...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

✅ Mursit: /root/.cache/huggingface/hub/models--boranyzgc--mursit-large-tur2-v2/snapshots/213f118fc39a30b3d5659725fbdee626ee74a1c7


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ BGE Reranker: /root/.cache/huggingface/hub/models--boranyzgc--bge-reranker-ft/snapshots/0feaa8012503ece01fe692a3b835de4b3c540b8b


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Trendyol adapter: /root/.cache/huggingface/hub/models--boranyzgc--trendyol-ft-context/snapshots/83e2d3246afa287b766a25de427cbbd9859013a3


mevzuat_chunked0v2_normalized.json:   0%|          | 0.00/3.32M [00:00<?, ?B/s]

✅ Mevzuat: /root/.cache/huggingface/hub/datasets--boranyzgc--turkish-legal-mevzuat/snapshots/a574d61cb855705d7beda6ca7e20cbf31510b927/mevzuat_chunked0v2_normalized.json


In [ ]:
import json, faiss, numpy as np, torch, re
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel
import gradio as gr

def pdf_oku(dosya_yolu):
    import pdfplumber
    metin = ""
    with pdfplumber.open(dosya_yolu) as pdf:
        for sayfa in pdf.pages:
            sayfa_metni = sayfa.extract_text()
            if sayfa_metni:
                metin += sayfa_metni + "\n\n"
    return metin

def txt_oku(dosya_yolu):
    with open(dosya_yolu, 'r', encoding='utf-8') as f:
        return f.read()

def genel_chunk(metin, kaynak_adi, chunk_size=400, overlap=50):
    # Temizle
    metin = re.sub(r'\n{3,}', '\n\n', metin)
    metin = re.sub(r' {2,}', ' ', metin)

    # Tekrar eden header satırlarını temizle
    satirlar = metin.split('\n')
    temiz_satirlar = []
    onceki = None
    for satir in satirlar:
        satir = satir.strip()
        # Tarih/saat içeren satırları atla
        if re.search(r'\d{2}\.\d{2}\.\d{4}\s+\d{2}:\d{2}', satir):
            continue
        # "Mevzuat Bilgi Sistemi" satırını atla
        if 'Mevzuat Bilgi Sistemi' in satir:
            continue
        # Tekrar eden satırları atla
        if satir == onceki:
            continue
        temiz_satirlar.append(satir)
        onceki = satir

    metin = '\n'.join(temiz_satirlar)
    metin = re.sub(r'\n{3,}', '\n\n', metin).strip()

    # Madde bazlı böl (varsa)
    madde_pattern = re.split(r'(?=Madde\s+\d+[-–])', metin)

    if len(madde_pattern) > 3:
        # Madde bazlı chunk
        chunks = []
        for i, bolum in enumerate(madde_pattern):
            bolum = bolum.strip()
            if not bolum or len(bolum.split()) < 10:
                continue
            # Büyük bölümleri böl
            if len(bolum.split()) > chunk_size:
                kelimeler = bolum.split()
                for j in range(0, len(kelimeler), chunk_size - overlap):
                    alt = ' '.join(kelimeler[j:j+chunk_size])
                    chunks.append({
                        "text": f"{kaynak_adi}\n\n{alt}",
                        "metadata": {
                            "chunk_id": f"{kaynak_adi}__bolum_{i}_{j}",
                            "kaynak": kaynak_adi,
                            "paragraf_no": str(i)
                        }
                    })
            else:
                chunks.append({
                    "text": f"{kaynak_adi}\n\n{bolum}",
                    "metadata": {
                        "chunk_id": f"{kaynak_adi}__bolum_{i}",
                        "kaynak": kaynak_adi,
                        "paragraf_no": str(i)
                    }
                })
        return chunks

    # Madde bulunamazsa paragraf bazlı
    paragraflar = [p.strip() for p in metin.split('\n\n') if len(p.strip()) > 20]
    chunks = []
    chunk_idx = 0
    mevcut_chunk = []
    mevcut_token = 0

    for paragraf in paragraflar:
        paragraf_token = len(paragraf.split())
        if paragraf_token > chunk_size:
            if mevcut_chunk:
                chunks.append({"text": f"{kaynak_adi}\n\n" + ' '.join(mevcut_chunk), "metadata": {"chunk_id": f"{kaynak_adi}__paragraf_{chunk_idx}", "kaynak": kaynak_adi, "paragraf_no": str(chunk_idx)}})
                chunk_idx += 1
                mevcut_chunk, mevcut_token = [], 0
            kelimeler = paragraf.split()
            for i in range(0, len(kelimeler), chunk_size - overlap):
                alt = ' '.join(kelimeler[i:i+chunk_size])
                chunks.append({"text": f"{kaynak_adi}\n\n{alt}", "metadata": {"chunk_id": f"{kaynak_adi}__paragraf_{chunk_idx}", "kaynak": kaynak_adi, "paragraf_no": str(chunk_idx)}})
                chunk_idx += 1
            continue
        if mevcut_token + paragraf_token <= chunk_size:
            mevcut_chunk.append(paragraf)
            mevcut_token += paragraf_token
        else:
            if mevcut_chunk:
                chunks.append({"text": f"{kaynak_adi}\n\n" + ' '.join(mevcut_chunk), "metadata": {"chunk_id": f"{kaynak_adi}__paragraf_{chunk_idx}", "kaynak": kaynak_adi, "paragraf_no": str(chunk_idx)}})
                chunk_idx += 1
            mevcut_chunk = mevcut_chunk[-1:] if overlap > 0 and mevcut_chunk else []
            mevcut_token = len(mevcut_chunk[0].split()) if mevcut_chunk else 0
            mevcut_chunk.append(paragraf)
            mevcut_token += paragraf_token

    if mevcut_chunk:
        chunks.append({"text": f"{kaynak_adi}\n\n" + ' '.join(mevcut_chunk), "metadata": {"chunk_id": f"{kaynak_adi}__paragraf_{chunk_idx}", "kaynak": kaynak_adi, "paragraf_no": str(chunk_idx)}})

    return chunks

In [ ]:
SYSTEM_PROMPT = (
    "Sen bir Türk hukuku uzmanı asistanısın. "
    "Cevaplarını SADECE verilen kanun metnine dayandır. "
    "Kanun metninde olmayan bilgileri ekleme. "
    "Her cevabın sonunda ilgili kanun maddesini belirt."
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Embedding modeli yükleniyor...")
embedding_model = SentenceTransformer(mursit_path)

print("BGE Reranker yükleniyor...")
reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_path)
reranker_model = AutoModelForSequenceClassification.from_pretrained(reranker_path)
reranker_model.eval().to('cuda')

print("LLM yükleniyor...")
llm_tokenizer = AutoTokenizer.from_pretrained('Trendyol/Trendyol-LLM-8b-chat-v2.0')
llm_tokenizer.pad_token = llm_tokenizer.eos_token
base_llm = AutoModelForCausalLM.from_pretrained(
    'Trendyol/Trendyol-LLM-8b-chat-v2.0',
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
llm_model = PeftModel.from_pretrained(base_llm, adapter_path)
llm_model.eval()

print(f"✅ Tüm modeller hazır. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Embedding modeli yükleniyor...


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

BGE Reranker yükleniyor...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

LLM yükleniyor...


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

✅ Tüm modeller hazır. GPU: 9.76 GB


In [ ]:
with open(mevzuat_path, 'r') as f:
    mevzuat_chunks = json.load(f)

mevzuat_texts = [c['text'] for c in mevzuat_chunks]

print("Mevzuat encode ediliyor...")
mevzuat_emb = embedding_model.encode(mevzuat_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
mevzuat_index = faiss.IndexFlatIP(mevzuat_emb.shape[1])
mevzuat_index.add(mevzuat_emb)

aktif_chunks = mevzuat_chunks
aktif_index = mevzuat_index
print(f"✅ Mevzuat index hazır: {mevzuat_index.ntotal} chunk")

Mevzuat encode ediliyor...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


✅ Mevzuat index hazır: 2324 chunk


In [ ]:
def rerank(soru, chunk_listesi, batch_size=16):
    skorlar = []
    for i in range(0, len(chunk_listesi), batch_size):
        batch = chunk_listesi[i:i+batch_size]
        enc = reranker_tokenizer(
            [soru]*len(batch), batch,
            truncation=True, max_length=512,
            padding=True, return_tensors='pt'
        ).to('cuda')
        with torch.no_grad():
            out = reranker_model(**enc)
            bs = out.logits.squeeze(-1).cpu().tolist()
            if isinstance(bs, float): bs = [bs]
            skorlar.extend(bs)
    return skorlar

def generate_answer(soru, context, max_new_tokens=256):
    maddeler = re.findall(r'MADDE\s+(\d+)', context)
    madde_str = ', '.join([f'Madde {m}' for m in maddeler[:3]]) if maddeler else ''
    user_content = (
        f"Aşağıdaki kanun maddesine dayanarak soruyu yanıtla:\n\n"
        f"{context}\n\n"
        f"Soru: {soru}\n\n"
        f"Önemli: Cevabının sonunda mutlaka 'Kaynak: [Kanun Adı] {madde_str}' yaz."
    )
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = llm_tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=llm_tokenizer.eos_token_id,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    cevap = llm_tokenizer.decode(generated, skip_special_tokens=True)
    if maddeler and 'madde' not in cevap.lower():
        cevap += f"\n\nKaynak: TCK {madde_str}"
    return cevap

def rag_pipeline(soru, chunks, index, top_k=20):
    soru_emb = embedding_model.encode([soru], normalize_embeddings=True)
    _, indices = index.search(soru_emb, top_k)
    top_k_idxler = indices[0].tolist()
    top_k_chunks = [chunks[idx]['text'] for idx in top_k_idxler]
    skorlar = rerank(soru, top_k_chunks)
    sirali = sorted(zip(top_k_idxler, top_k_chunks, skorlar), key=lambda x: x[2], reverse=True)
    top3 = [c for _, c, _ in sirali[:3]]
    context = "\n\n---\n\n".join(top3)
    cevap = generate_answer(soru, context)
    return cevap, context, top3

print("✅ Pipeline hazır")

✅ Pipeline hazır


In [ ]:
ozel_chunks = None
ozel_index = None

def dokuman_yukle(dosya):
    global ozel_chunks, ozel_index
    if dosya is None:
        return "❌ Dosya seçilmedi"
    try:
        chunks = dokuman_isle(dosya.name)
        texts = [c['text'] for c in chunks]
        emb = embedding_model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
        idx = faiss.IndexFlatIP(emb.shape[1])
        idx.add(emb)
        ozel_chunks = chunks
        ozel_index = idx
        return f"✅ {Path(dosya.name).name} yüklendi — {len(chunks)} chunk oluşturuldu"
    except Exception as e:
        return f"❌ Hata: {str(e)}"

def soru_sor(soru, mod):
    if not soru.strip():
        return "Lütfen bir soru girin.", ""
    try:
        if mod == "Hazır Mevzuat (5 Kanun)":
            cevap, context, top3 = rag_pipeline(soru, aktif_chunks, aktif_index)
        else:
            if ozel_chunks is None:
                return "❌ Önce bir döküman yükleyin.", ""
            cevap, context, top3 = rag_pipeline(soru, ozel_chunks, ozel_index)
        kaynaklar = "\n\n".join([f"**Kaynak {i+1}:**\n{c[:300]}..." for i, c in enumerate(top3)])
        return cevap, kaynaklar
    except Exception as e:
        return f"❌ Hata: {str(e)}", ""

with gr.Blocks(title="Türk Hukuku RAG Sistemi", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚖️ Türk Hukuku RAG Sistemi")
    gr.Markdown("**Fine-tuned Embedding + BGE Reranker + Fine-tuned LLM** | TCK, TMK, İş Kanunu, TBK, Anayasa")

    with gr.Row():
        with gr.Column(scale=1):
            mod = gr.Radio(
                choices=["Hazır Mevzuat (5 Kanun)", "Özel Döküman"],
                value="Hazır Mevzuat (5 Kanun)",
                label="🗂️ Mod Seçin"
            )
            dosya = gr.File(label="📄 PDF veya TXT Yükle", file_types=[".pdf", ".txt"], visible=False)
            yukle_btn = gr.Button("📥 Dökümanı Yükle", visible=False)
            yukle_sonuc = gr.Textbox(label="Yükleme Durumu", visible=False)

            def mod_degisti(mod):
                gorunur = mod == "Özel Döküman"
                return gr.update(visible=gorunur), gr.update(visible=gorunur), gr.update(visible=gorunur)

            mod.change(mod_degisti, inputs=mod, outputs=[dosya, yukle_btn, yukle_sonuc])
            yukle_btn.click(dokuman_yukle, inputs=dosya, outputs=yukle_sonuc)

        with gr.Column(scale=2):
            soru_input = gr.Textbox(
                label="❓ Sorunuz",
                placeholder="Örn: Kasten öldürme suçunun cezası nedir?",
                lines=3
            )
            sor_btn = gr.Button("🔍 Sor", variant="primary", size="lg")
            cevap_output = gr.Textbox(label="💬 Cevap", lines=6)
            kaynaklar_output = gr.Textbox(label="📚 Kullanılan Kaynaklar", lines=5)

    gr.Markdown("---\n*Bu sistem TCK, TMK, İş Kanunu, TBK ve Anayasa üzerine fine-tune edilmiştir. Hukuki tavsiye niteliği taşımaz.*")

    sor_btn.click(soru_sor, inputs=[soru_input, mod], outputs=[cevap_output, kaynaklar_output])

demo.launch(share=True, debug=False)

/tmp/ipykernel_2084/1322190796.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Türk Hukuku RAG Sistemi", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3887f2946215d583ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
